In [1]:
import requests
import pandas as pd
import os

In [9]:
import sys
print(sys.executable)   # la ruta exacta al binario de Python que usa el kernel
print(sys.version)      # la versión de Python

/usr/local/bin/python3
3.13.7 (v3.13.7:bcee1c32211, Aug 14 2025, 19:10:51) [Clang 16.0.0 (clang-1600.0.26.6)]


In [ ]:
"""
Script exploratorio: extrae la tabla de posiciones del Mundial desde
API-Football y arma un DataFrame simple: pais, puntos, mundial, grupo.

Instalar dependencias antes de correr:
    pip install requests pandas

Correr con:
    python extract_standings.py
"""

API_KEY = os.getenv("API_FOOTBALL_KEY")   # reemplazar por tu key real
LEAGUE_ID = 1                      # 1 = World Cup
SEASON = 2022                     # cambiar a 2022 si querés probar con datos ya completos

BASE_URL = "https://v3.football.api-sports.io/standings"
 
 
def extract_standings(league_id: int, season: int) -> pd.DataFrame:
    """Pega a la API y devuelve un DataFrame plano: pais, puntos, mundial, grupo."""
    headers = {"x-apisports-key": API_KEY}
    params = {"league": league_id, "season": season}
 
    response = requests.get(BASE_URL, headers=headers, params=params)
    response.raise_for_status()   # si hay error HTTP, corta acá con un mensaje claro
    data = response.json()
 
    # --- DEBUG: ver la respuesta cruda antes de parsear ---
    print("--- Respuesta cruda de la API ---")
    print(data.get("errors"))
    print(f"Cantidad de resultados (results): {data.get('results')}")
    print(f"Response (primeros 500 caracteres): {str(data.get('response'))[:500]}")
    print("--- fin debug ---\n")
 
    rows = []
    for standings_response in data["response"]:
        league = standings_response["league"]
        for group in league["standings"]:
            for team_row in group:
                rows.append({
                    "pais": team_row["team"]["name"],
                    "puntos": team_row["points"],
                    "mundial": league["season"],
                    "grupo": team_row["group"],
                })
 
    return pd.DataFrame(rows)
 
 
if __name__ == "__main__":
    df = extract_standings(LEAGUE_ID, SEASON)
 
    print(f"\nSe extrajeron {len(df)} filas.\n")
    print(df.head(10))
 
    # Un par de vistazos extra, útiles para entender la forma de los datos
    print("\n--- Tipos de datos ---")
    print(df.dtypes)
 
    print("\n--- Agrupado por grupo (ordenado por puntos) ---")
    print(df.sort_values(["grupo", "puntos"], ascending=[True, False]))

--- Respuesta cruda de la API ---
[]
Cantidad de resultados (results): 1
Response (primeros 500 caracteres): [{'league': {'id': 1, 'name': 'World Cup', 'country': 'World', 'logo': 'https://media.api-sports.io/football/leagues/1.png', 'flag': None, 'season': 2022, 'standings': [[{'rank': 1, 'team': {'id': 1118, 'name': 'Netherlands', 'logo': 'https://media.api-sports.io/football/teams/1118.png'}, 'points': 7, 'goalsDiff': 4, 'group': 'Group A', 'form': 'WDW', 'status': 'same', 'description': 'Promotion - World Cup (Play Offs)', 'all': {'played': 3, 'win': 2, 'draw': 1, 'lose': 0, 'goals': {'for': 5, 'ag
--- fin debug ---


Se extrajeron 32 filas.

          pais  puntos  mundial    grupo
0  Netherlands       7     2022  Group A
1      Senegal       6     2022  Group A
2      Ecuador       4     2022  Group A
3        Qatar       0     2022  Group A
4      England       7     2022  Group B
5          USA       5     2022  Group B
6         Iran       3     2022  Group B
7        Wales   